**Import Libraries**

In [110]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder

import matplotlib.pyplot as plt
import seaborn as sns

**Load the Dataset**

In [111]:
df = pd.read_csv("/content/drive/MyDrive/Heart Diseases - AI Project/datasets/Original_Datasets/healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [112]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


**Check if there are some null values**

In [113]:
df.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


In [114]:
df = df.dropna()

In [115]:
df.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,0


In [116]:
df = df.drop('id', axis=1)

**Check If there are NaN values**

In [117]:
df.isna().sum()

,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,0
smoking_status,0


**Encode Categorical Variables**


In [118]:
binary_cols = ['gender', 'ever_married']


multi_cols = ['work_type', 'Residence_type', 'smoking_status']


le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

ohe = OneHotEncoder(sparse_output=False, drop='first')

onehot_data = ohe.fit_transform(df[multi_cols])

onehot_df = pd.DataFrame(
    onehot_data,
    columns=ohe.get_feature_names_out(multi_cols),
    index=df.index   # 🔥 THIS FIXES EVERYTHING
)

df = df.drop(multi_cols, axis=1)
df = pd.concat([df, onehot_df], axis=1)

**Feature Engineering**

In [119]:
# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 30, 50, 65, 120],
    labels=["Young", "Adult", "Middle_Aged", "Elderly"]
)

# BMI categories
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# Risk score (important medical feature)
df["risk_score"] = (
    df["age"] * 0.3 +
    df["avg_glucose_level"] * 0.4 +
    df["hypertension"] * 2 +
    df["heart_disease"] * 2
)

**Encode NEW engineered features**

In [120]:
df = pd.get_dummies(df, columns=["age_group", "bmi_category"], drop_first=True)

**Define Features and Target**

In [121]:
X = df.drop('stroke', axis=1)
y = df['stroke']

**TRAIN/TEST SPLIT**

In [122]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

In [123]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

**Scaling Numerical Features**

In [126]:
scaler = StandardScaler()


X_train = scaler.fit_transform(X_train)


X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


X_train = pd.DataFrame(X_train, columns=X.columns)
X_val = pd.DataFrame(X_val, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

**Save the dataset**

In [129]:
np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("X_test.npy", X_test)

np.save("y_train.npy", y_train)
np.save("y_val.npy", y_val)
np.save("y_test.npy", y_test)

In [130]:
df.to_csv("stroke_clean_Fin.csv", index=False)